
# Caculation for RSSI 

Formule d=10^((ABS(RSSI)-A)/10*n)

- d=distance à l’émetteur 
- A=valeur absolue de la puissance à un mètre de distance à l ‘émetteur 
- n=coefficient de perte de trajet (n=2.2);
- RSSI=valeur de puissance reçu ;

 Sources : 
 https://projets-ima.plil.fr/mediawiki/index.php/Mesure_de_distance_par_RSSI#:~:text=D%27abord%2C%20on%20fait%20le,n

In [ ]:
import math
import subprocess


def calcul_distance(RSSI, A, n):
    """
    Calcule la distance à partir du RSSI en utilisant la formule :
    d = 10^((ABS(RSSI) - A) / (10 * n))
    
    Args:
    - RSSI (float): Valeur du RSSI (Received Signal Strength Indication).
    - A (float): Valeur absolue de la puissance à un mètre de distance à l'émetteur.
    - n (float): Coefficient de perte de trajet.
    
    Returns:
    - float: La distance calculée en mètres.
    """
    distance = 10 ** ((abs(RSSI) - A) / (10 * n))
    return distance
def calcul_distance_with_ip(ip):
    """
    Calcule la distance à partir du RSSI en utilisant la formule :
    d = 10^((ABS(RSSI) - A) / (10 * n))
    
    Args:
    - RSSI (float): Valeur du RSSI (Received Signal Strength Indication).
    - A (float): Valeur absolue de la puissance à un mètre de distance à l'émetteur.
    - n (float): Coefficient de perte de trajet.
    - ip (str): Adresse IP de l'émetteur.
    
    Returns:
    - float: La distance calculée en mètres.
    """
    # Ping (ms)
    ping_response = subprocess.Popen(["ping", "-c", "1", "-w", "1", ip], stdout=subprocess.PIPE).stdout.read()
    ping_response = ping_response.decode("utf-8")
    
    # Extract ms 
    latency_ms = float(ping_response.split("time=")[1].split(" ms")[0])
    
    # Convert ms to m 
    speed_of_light = 299792458 # m/s (vitesse de la lumière)
    distance = (latency_ms / 1000) * (speed_of_light / 2)
    
    return distance

# Exemple d'utilisation :
RSSI = -70  
A = 50     
n = 2.2    
ip = "0.0.0.0"  

distance = calcul_distance(RSSI, A, n)
distance_m_ip = calcul_distance_with_ip(ip)
print("Distance correspondante :", distance, "mètres")
print("Distance en mètre par rapport a l'ip correspondante :", distance_m_ip, "mètres")


# Triangulation 

Prérequie : 

- Distance de chacun des ESP32 par rapport au Serveur(Raspberry)


In [ ]:
# Fonction pour effectuer la triangulation à partir des données RSSI de trois émetteurs
def triangulation(rssi1, rssi2, rssi3, A, n):
    # Coordonnées des émetteurs
    x1, y1 = 0, 0  # Coordonnées de l'émetteur 1
    x2, y2 = 3, 0  # Coordonnées de l'émetteur 2
    x3, y3 = 0, 4  # Coordonnées de l'émetteur 3
    
    # Calcul des distances à partir des signaux RSSI
    d1 = calcul_distance(rssi1, A, n)
    d2 = calcul_distance(rssi2, A, n)
    d3 = calcul_distance(rssi3, A, n)
    
    # Triangulation
    x = (d1**2 - d2**2 + x2**2) / (2*x2)
    y = (d1**2 - d3**2 + d3**2 - 2*x3*x + x3**2 + y3**2) / (2*y3)
    
    return x, y

# Exemple d'utilisation :
rssi1 = -70  
rssi2 = -75  
rssi3 = -80  
A = 50      
n = 2.2     

x, y = triangulation(rssi1, rssi2, rssi3, A, n)
print("Coordonnées estimées (x, y) :", (x, y))


   # Calcule la valeur absolue de la puissance à un mètre de distance (A)


In [ ]:
def calcul_A(d1, d2, d3, RSSI1, RSSI2, RSSI3, n):
    """
    Calcule la valeur absolue de la puissance à un mètre de distance (A)
    à partir des distances mesurées (d1, d2, d3) et des signaux RSSI (RSSI1, RSSI2, RSSI3).

    Args:
    - d1 (float): Distance mesurée à partir de l'émetteur 1 (en mètres).
    - d2 (float): Distance mesurée à partir de l'émetteur 2 (en mètres).
    - d3 (float): Distance mesurée à partir de l'émetteur 3 (en mètres).
    - RSSI1 (float): Valeur du RSSI (Received Signal Strength Indication) pour l'émetteur 1.
    - RSSI2 (float): Valeur du RSSI (Received Signal Strength Indication) pour l'émetteur 2.
    - RSSI3 (float): Valeur du RSSI (Received Signal Strength Indication) pour l'émetteur 3.
    - n (float): Coefficient de perte de trajet.

    Returns:
    - float: La valeur absolue de la puissance à un mètre de distance (A).
    """
    A = (abs(RSSI1) - 10 * n * math.log10(d1) + abs(RSSI2) - 10 * n * math.log10(d2) + abs(RSSI3) - 10 * n * math.log10(d3)) / 3
    return A

# Exemple d'utilisation :
d1 = 1.5  
d2 = 2.5  
d3 = 3.5  
RSSI1 = -70  
RSSI2 = -75  
RSSI3 = -80  
n = 2.2    

A = calcul_A(d1, d2, d3, RSSI1, RSSI2, RSSI3, n)
print("Valeur absolue de la puissance à un mètre de distance (A) :", A)


# Regroup all function 

In [ ]:
# Fonction pour effectuer la triangulation à partir des données RSSI de trois émetteurs
def triangulation(rssi1, rssi2, rssi3, d1, d2, d3, n):
    # Calcul de la valeur absolue de la puissance à un mètre de distance (A)
    A = calcul_A(d1, d2, d3, rssi1, rssi2, rssi3, n)
    
    # Coordonnées des émetteurs
    x1, y1 = 0, 0  
    x2, y2 = 3, 0  
    x3, y3 = 0, 4 
    
    # Calcul des distances à partir des signaux RSSI
    d1 = calcul_distance(rssi1, A, n)
    d2 = calcul_distance(rssi2, A, n)
    d3 = calcul_distance(rssi3, A, n)
    
    # Triangulation
    x = (d1**2 - d2**2 + x2**2) / (2*x2)
    y = (d1**2 - d3**2 + d3**2 - 2*x3*x + x3**2 + y3**2) / (2*y3)
    
    return x, y

# RSSI a calculé avec la fonction a partir de l'adresse mac de ESP32 
rssi1 = -75  
rssi2 = -105  
rssi3 = -50 
# rssi1 = data[0]
# rssi2 = data[1] 
# rssi3 = data[2] 

#Distance fix 
d1 = 1.5  
d2 = 2.5  
d3 = 3.5 
# Distance en fonction l'ip
# d1 = calcul_distance_with_ip("0.0.0.0")
# d2 = calcul_distance_with_ip("1.1.1.1") 
# d3 = calcul_distance_with_ip("2.2.2.2") 
#Valeur fix 
n = 2.2   

x, y = triangulation(rssi1, rssi2, rssi3, d1, d2, d3, n)
print("Coordonnées estimées (x, y) :", (x, y))
